In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Load sample dataset and features
sample_df = pd.read_csv("../data/sample_5000.csv")
image_features = np.load("../data/image_features.npy")

print("Sample shape:", sample_df.shape)
print("Image features shape:", image_features.shape)

In [ ]:
# Text preprocessing and TF-IDF vectorization
sample_df["catalog_content"] = (
    sample_df["catalog_content"].astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_text = vectorizer.fit_transform(sample_df["catalog_content"])
print("TF-IDF shape:", X_text.shape)

In [ ]:
# Combine text and image features into a multimodal feature matrix
X_final = hstack([X_text, csr_matrix(image_features)])
print("Combined multimodal shape:", X_final.shape)

In [ ]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    sample_df["price"],
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# Train multimodal regression model
model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

In [ ]:
# Evaluate multimodal model
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
r2 = r2_score(y_test, preds)

print("Multimodal MAE:", mae)
print("Multimodal RMSE:", rmse)
print("Multimodal R²:", r2)